# AcrPred 严格复现 (Dao et al., IJBM 2023)

**方法**：混合优化 + 枚举式 SVM 集成

**特征**（4 个独立特征组）：
- CTD (168d)：8 种理化性质的 Composition-Transition-Distribution（源码 `feature188d.py` 实际输出 168d，AAC 被注释掉）
- PSSM Composition (400d)：按氨基酸类型统计 PSSM 列均值
- DPC-PSSM (400d)：相邻位置 PSSM 列的二肽相关
- PSSM-AC (200d, lag=10)：PSSM 列的自相关

**模型结构**（严格对齐论文与源码 `predict_main.py`）：
1. 4 个特征组**分别**标准化（scaler 在本实验的 train split 上 fit）
2. 每个特征组使用**原始仓库硬编码的特征选择索引**（`para` 变量，源自原文 hybrid optimization 的预计算结果）
3. 对训练集做 5 次**随机欠采样**（平衡正负样本）
4. 每个特征组 × 5 次采样 = 20 个 SVM (RBF kernel)
5. 最终预测 = 20 个 SVM 概率输出的**平均值**

**公平比较设计**：特征选择索引视为**方法论层面的设计**（类似网络架构/超参数），直接复用；
scaler 和 SVM 模型在本实验的 train split 上重新训练，确保与其他方法在同一数据划分下公平评估。

**参考**：https://github.com/linDing-group/AcrPred

**环境**：`lm-hf`

In [1]:
import os, json, glob
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score,
)

BENCHMARKS_DIR = '/home/nemophila/projects/protein_bert/anticrispr_benchmarks'
PSSM_DIR       = '/home/nemophila/data/pssm_work/pssm'
MANIFEST_PATH  = '/home/nemophila/data/pssm_work/sample_manifest.csv'
RESULTS_DIR    = '/home/nemophila/projects/protein_bert/Comparison/results'
SEED = 22
N_SAMPLINGS = 5  # 原文: 5 次平衡采样

## 1. 加载数据

In [2]:
train_df = pd.read_csv(f'{BENCHMARKS_DIR}/anticrispr_binary.train.csv').dropna().drop_duplicates().reset_index(drop=True)
test_df  = pd.read_csv(f'{BENCHMARKS_DIR}/anticrispr_binary.test.csv').dropna().drop_duplicates().reset_index(drop=True)
train_df['sample_id'] = [f'train_{i:06d}' for i in range(len(train_df))]
test_df['sample_id']  = [f'test_{i:06d}' for i in range(len(test_df))]

manifest = pd.read_csv(MANIFEST_PATH)
print('train:', train_df.shape, 'test:', test_df.shape)

train: (1107, 3) test: (286, 3)


## 2. PSSM 文件解析

解析 PSI-BLAST 标准 PSSM 格式，提取 L×20 评分矩阵。

In [3]:
AA_ORDER = list('ARNDCQEGHILKMFPSTWYV')  # PSI-BLAST 标准顺序


def parse_pssm_file(filepath: str):
    """解析 PSI-BLAST .pssm 文件，返回 (sequence, pssm_matrix)。
    pssm_matrix: shape (L, 20)，每行为该位置的 20 个 PSSM 得分。
    """
    with open(filepath) as f:
        lines = f.readlines()

    seq_chars = []
    scores = []
    for line in lines[3:]:  # 跳过前 3 行头部
        line = line.rstrip()
        if not line or line.startswith('\n'):
            break
        parts = line.split()
        if len(parts) < 22:
            break
        try:
            int(parts[0])  # 位置编号
        except ValueError:
            break
        seq_chars.append(parts[1])
        scores.append([int(x) for x in parts[2:22]])

    return ''.join(seq_chars), np.array(scores, dtype=np.float64)


# 测试
test_file = os.path.join(PSSM_DIR, 'test_000000.pssm')
seq_test, pssm_test = parse_pssm_file(test_file)
print(f'Test parse: seq len={len(seq_test)}, pssm shape={pssm_test.shape}')

Test parse: seq len=124, pssm shape=(124, 20)


## 3. AcrPred 特征提取函数

严格对齐 `feature188d.py`（CTD 168d）和 `predict_main.py`（PSSM 特征）。

**注意**：`feature188d.py` 中 AAC 代码被注释掉，实际输出 8×(3C+3T+15D) = 168 维。

In [4]:
# ---------- CTD 168d (从 feature188d.py 严格移植) ---------- #
# 源码开头注释: "八个理化性质分成3类，维度168"
# 函数名 feature188d 具有误导性 — AAC 20d 代码被注释掉，实际输出 168d

PP = [
    [['R','K','E','D','Q','N'], ['G','A','S','T','P','H','Y'], ['C','L','V','I','M','F','W']],
    [['G','A','S','T','P','D','C'], ['N','V','E','Q','I','L'], ['M','H','K','F','R','Y','W']],
    [['L','I','F','W','C','M','V','Y'], ['P','A','T','G','S'], ['H','Q','R','K','N','E','D']],
    [['G','A','S','D','T'], ['C','P','N','V','E','Q','I','L'], ['K','M','H','F','R','Y','W']],
    [['K','R'], ['A','N','C','Q','G','H','I','L','M','F','P','S','T','W','Y','V'], ['D','E']],
    [['E','A','L','M','Q','K','R','H'], ['V','I','Y','C','W','F','T'], ['G','N','P','S','D']],
    [['A','L','F','C','G','I','V','W'], ['P','K','Q','E','N','D'], ['M','R','S','T','H','Y']],
    [['G','Q','D','N','A','H','R'], ['K','T','S','E','C'], ['I','L','M','F','P','W','Y','V']],
]


def ctd_168d(seq: str) -> np.ndarray:
    """CTD 特征 168d：8 种理化性质 × (3C + 3T + 15D) = 168。
    严格按照 feature188d.py 的逻辑移植。
    """
    L = len(seq)
    ctd = []
    for j in range(8):
        n1 = np.zeros(3)
        n2 = np.zeros(3)  # transitions: (0↔1), (0↔2), (1↔2)
        n3 = [[], [], []]

        for k in range(L):
            for g in range(3):
                if seq[k] in PP[j][g]:
                    n1[g] += 1
                    n3[g].append(k + 1)
                    if k + 1 < L:
                        if g == 0:
                            if seq[k + 1] in PP[j][1]: n2[0] += 1
                            elif seq[k + 1] in PP[j][2]: n2[1] += 1
                        elif g == 1:
                            if seq[k + 1] in PP[j][0]: n2[0] += 1
                            elif seq[k + 1] in PP[j][2]: n2[2] += 1
                        elif g == 2:
                            if seq[k + 1] in PP[j][0]: n2[1] += 1
                            elif seq[k + 1] in PP[j][1]: n2[2] += 1
                    break

        # C: composition (3)
        c = n1 / L
        # T: transition (3)
        t = n2 / max(L - 1, 1)
        # D: distribution (3 groups × 5 quantiles) = 15
        d = np.zeros((3, 5))
        for g in range(3):
            cnt = int(n1[g])
            if cnt > 0:
                d[g][0] = n3[g][0] / L
            if cnt >= 4:
                d[g][1] = n3[g][int(0.25 * cnt) - 1] / L
            if cnt >= 2:
                d[g][2] = n3[g][int(0.5 * cnt) - 1] / L
            if cnt >= 2:
                d[g][3] = n3[g][int(0.75 * cnt) - 1] / L
            if cnt > 0:
                d[g][4] = n3[g][cnt - 1] / L

        # 原始代码顺序: 先所有 C, 再所有 T, 再所有 D
        ctd.extend(c.tolist())
        ctd.extend(t.tolist())
        ctd.extend(d.flatten().tolist())

    return np.array(ctd, dtype=np.float64)  # 8 × 21 = 168


print('CTD dim:', len(ctd_168d('ARNDCQEGHILKMFPSTWYV')))

CTD dim: 168


In [5]:
# ---------- PSSM-based features (从 predict_main.py 严格移植) ---------- #

def pssm_composition(seq: str, pssm: np.ndarray) -> np.ndarray:
    """PSSM Composition (400d): 与原始代码完全一致。"""
    aa_set = list('ARNDCQEGHILKMFPSTWYV')
    L = len(seq)
    aa_dict = {}
    for i, aa in enumerate(seq):
        if aa not in aa_dict:
            aa_dict[aa] = np.zeros(20)
        aa_dict[aa] += pssm[i] / L
    result = []
    for aa in aa_set:
        for j in range(20):
            result.append(aa_dict[aa][j] if aa in aa_dict else 0.0)
    return np.array(result, dtype=np.float64)


def dpc_pssm(pssm: np.ndarray) -> np.ndarray:
    """DPC-PSSM (400d): 与原始代码完全一致。"""
    L = pssm.shape[0]
    result = []
    for i in range(20):
        for j in range(20):
            val = sum(pssm[k, i] * pssm[k + 1, j] for k in range(L - 1))
            result.append(val / max(L - 1, 1))
    return np.array(result, dtype=np.float64)


def pssm_ac(pssm: np.ndarray, lg: int = 10) -> np.ndarray:
    """PSSM-AC (20*lg d): 与原始代码完全一致。"""
    L = pssm.shape[0]
    col_mean = pssm.mean(axis=0)
    result = []
    for j in range(20):
        for lag in range(1, lg + 1):
            val = sum((pssm[i, j] - col_mean[j]) * (pssm[i + lag, j] - col_mean[j])
                      for i in range(L - lag))
            result.append(val / max(L - lag, 1))
    return np.array(result, dtype=np.float64)


def extract_four_groups(seq: str, pssm: np.ndarray):
    """返回 4 个独立特征组（与原始 predict_main.py 的 feature_extraction 对齐）。"""
    g1 = ctd_168d(seq)           # 168d
    g2 = pssm_composition(seq, pssm)  # 400d
    g3 = dpc_pssm(pssm)         # 400d
    g4 = pssm_ac(pssm, lg=10)   # 200d
    return g1, g2, g3, g4


# 验证维度
g1, g2, g3, g4 = extract_four_groups(seq_test, pssm_test)
print(f'Group dims: CTD={len(g1)}, PSSM-co={len(g2)}, DPC-PSSM={len(g3)}, PSSM-AC={len(g4)}')

Group dims: CTD=168, PSSM-co=400, DPC-PSSM=400, PSSM-AC=200


In [6]:
# 批量提取：返回 4 个独立特征组的列表
def batch_extract_groups(df, pssm_dir, split_name):
    groups = [[], [], [], []]  # 4 个特征组
    valid_idx = []
    for i in range(len(df)):
        sid = f'{split_name}_{i:06d}'
        pssm_path = os.path.join(pssm_dir, f'{sid}.pssm')
        if not os.path.exists(pssm_path):
            continue
        seq_parsed, pssm_mat = parse_pssm_file(pssm_path)
        if pssm_mat.shape[0] < 11:  # PSSM-AC 需要 lag+1=11
            continue
        g1, g2, g3, g4 = extract_four_groups(seq_parsed, pssm_mat)
        feats = [g1, g2, g3, g4]
        if any(np.any(np.isnan(f)) or np.any(np.isinf(f)) for f in feats):
            continue
        for j in range(4):
            groups[j].append(feats[j])
        valid_idx.append(i)
    return [np.array(g) for g in groups], valid_idx


print('Extracting training features (4 groups)...')
train_groups, train_valid_idx = batch_extract_groups(train_df, PSSM_DIR, 'train')
print(f'  Train: {train_groups[0].shape[0]}/{len(train_df)} samples')
for i, name in enumerate(['CTD', 'PSSM-co', 'DPC-PSSM', 'PSSM-AC']):
    print(f'    Group {i+1} ({name}): {train_groups[i].shape}')

print('Extracting test features (4 groups)...')
test_groups, test_valid_idx = batch_extract_groups(test_df, PSSM_DIR, 'test')
print(f'  Test: {test_groups[0].shape[0]}/{len(test_df)} samples')

y_train_all = train_df.iloc[train_valid_idx]['label'].to_numpy(dtype=int)
y_test  = test_df.iloc[test_valid_idx]['label'].to_numpy(dtype=int)

Extracting training features (4 groups)...
  Train: 1107/1107 samples
    Group 1 (CTD): (1107, 168)
    Group 2 (PSSM-co): (1107, 400)
    Group 3 (DPC-PSSM): (1107, 400)
    Group 4 (PSSM-AC): (1107, 200)
Extracting test features (4 groups)...
  Test: 286/286 samples


## 4. 训练 20 SVM 集成模型

严格对齐 `predict_main.py` 的 `perform_prediction` 结构：
- 4 个特征组 × 5 次平衡采样 = **20 个 SVM**
- 每个特征组单独做 StandardScaler（在本实验 train split 上 fit）
- **特征选择**：直接使用原始仓库 `predict_main.py` 中的 `para` 变量（硬编码索引）
  - 该变量是原文 hybrid optimization 的预计算结果，属于方法论设计的一部分
  - 索引为 **1-based**，使用时需 -1 转换为 0-based
- 每次采样通过随机欠采样多数类实现正负平衡
- SVM 超参数通过 **GridSearchCV** 对每个特征组优化（搜索 C, gamma）
- 最终预测 = 20 个模型概率输出的平均值

In [7]:
GROUP_NAMES = ['CTD', 'PSSM-co', 'DPC-PSSM', 'PSSM-AC']

# --- Step 1: 每个特征组单独标准化（与原始 feature_extraction 中 4 个 scaler 对应） ---
scalers = []
train_groups_scaled = []
test_groups_scaled = []

for i in range(4):
    sc = StandardScaler()
    tr = sc.fit_transform(train_groups[i])
    te = sc.transform(test_groups[i])
    tr = np.nan_to_num(tr, nan=0.0, posinf=0.0, neginf=0.0)
    te = np.nan_to_num(te, nan=0.0, posinf=0.0, neginf=0.0)
    scalers.append(sc)
    train_groups_scaled.append(tr)
    test_groups_scaled.append(te)
    print(f'Group {i+1} ({GROUP_NAMES[i]}): scaled, dim={tr.shape[1]}')


# ===== 原始仓库 predict_main.py 中的硬编码特征选择索引 (para) =====
# 来源: https://github.com/linDing-group/AcrPred/blob/main/predict_main.py
# 结构: PARA[group_i][sampling_j] = 1-based 特征索引列表
# group 0 = CTD (168d), group 1 = PSSM-co (400d), group 2 = DPC-PSSM (400d), group 3 = PSSM-AC (200d)
# 索引排序后使用: para[i][j].sort(); feature[indexj - 1]
PARA = [[[15, 39, 43, 32, 21, 20, 26, 8, 2, 44, 27, 31, 1, 88, 113, 14, 9, 123, 30, 85, 16, 60, 152, 17, 57, 37, 153, 42, 13, 98, 68, 122, 58, 148, 140, 62, 4, 36, 102, 157, 124, 137, 87, 65, 49, 150, 119, 55, 77, 107, 80, 5, 120, 149, 114, 112, 72, 82, 41, 7, 81, 133, 167, 93, 78, 108, 160, 48, 95, 115, 53, 10, 19, 73, 86, 165, 33, 151, 89, 23, 61, 110, 28, 84, 45, 56, 54, 22, 103, 97], [15, 32, 88, 58, 26, 20, 8, 2, 39, 21, 152, 113, 57, 1, 68, 43, 44, 30, 4, 17, 87, 102, 153, 9, 73, 98, 27, 5, 123, 31, 48, 84, 72, 7, 137, 14, 33, 115, 36, 122, 10, 120, 103, 22, 37, 138, 127, 78, 108, 64, 54, 41, 42, 148, 76, 106, 112, 16, 158, 124, 85, 13, 160, 114, 23, 121, 40, 157, 25, 71, 154, 110, 74, 104, 11, 151, 149, 97, 93, 59, 67, 131, 77, 107, 19, 111, 164, 79, 118, 150, 128, 86, 53, 101, 3, 94, 134, 147, 56, 29, 55, 139, 119, 132, 140, 100, 161, 92, 141, 167, 62, 34, 70, 129, 6, 12, 46, 75, 105, 50], [15, 39, 20, 44, 21, 43, 2, 8, 26, 32, 27, 1, 113, 31, 102, 37, 16, 120, 84, 17, 72, 14, 9, 13, 123, 88, 57, 58, 30, 42, 152, 5, 4, 7, 167, 71, 87, 76, 106, 153, 67, 97, 64, 157, 54, 22, 48, 112, 111, 101, 77, 107, 151, 148, 36, 94, 62, 150, 73, 68, 114, 78, 108, 127, 138, 134, 115, 82, 19, 133, 119, 122, 110, 46, 154, 41, 10, 103, 144, 128, 23, 3, 129, 161, 24, 145, 75, 105, 11, 146, 121, 98, 91, 160, 53, 136, 86, 55, 85, 45, 63, 93, 137, 18, 156, 140, 51, 141, 131, 126], [15, 39, 43, 32, 26, 21, 20, 27, 8, 31, 2, 88, 58, 44, 113, 1, 102, 30, 14, 41, 123, 119, 160, 9, 37, 4, 157, 17, 68, 134, 167, 84, 13, 140, 42, 7, 98, 5, 60, 97, 153, 72, 152, 148, 36, 87, 137, 128, 80, 57, 77, 107, 112, 22, 16, 53, 110, 64, 165, 62, 93, 67, 124, 150, 154, 45, 34, 85, 54, 10, 114, 138, 161, 129, 95, 23, 28, 76, 106, 111, 19, 166, 48, 52, 82, 159, 6, 12, 118, 151, 78, 108, 133, 59, 127, 86, 33, 156, 81, 71, 3, 164, 79, 101, 96, 120, 144, 163, 90, 131, 126, 135, 92, 141, 130, 29, 91, 50, 65, 24, 100, 51, 158, 122, 115, 55, 75, 105, 83, 11, 168, 94, 38, 63, 74, 104, 40, 155, 125, 35, 25, 66, 121, 145, 69, 162, 89, 116, 46, 143, 18, 61, 136, 56, 117, 132, 103, 99, 146, 139], [15, 39, 16, 17, 113, 88, 42, 102, 20, 43, 37, 21, 58, 32, 110, 8, 13, 26, 72, 77, 107, 2, 27, 57, 123, 112, 68, 31, 140, 62]], [[206, 64, 33, 207, 350, 353, 126, 351, 35, 342, 215, 37, 325, 204, 335, 28, 31, 25, 24, 36, 34, 352, 205, 286, 288, 15, 40, 27, 61, 360, 305, 349, 284, 144, 203, 281, 307, 285, 212, 73, 78, 23, 202, 346, 340, 379, 330, 341, 71, 357, 62, 147, 292, 217, 347, 21, 216, 209, 287, 38, 356, 208, 331, 30, 320, 345, 343, 115, 374, 386, 145, 80, 39, 227, 70, 5, 127, 369, 22, 333, 324, 69, 279, 283, 334, 348, 323, 44, 299, 304, 246, 308, 300, 319, 294, 313, 74, 124, 235, 327, 289, 57, 306, 328, 339, 155, 253, 47, 167, 274, 293, 67, 79, 175, 136, 132, 315, 156, 56, 387, 355, 113, 116, 7, 392, 105, 383, 314, 329, 120, 211, 338, 271, 311, 219, 291, 157, 318, 185, 303, 201, 310, 197, 298, 135, 254, 143, 395, 153, 200, 295, 385, 4, 370, 384, 290, 123, 218, 344, 20, 396, 312, 8, 130, 165, 236, 146, 149, 119, 282, 111, 378, 160, 43, 102, 134, 65, 118, 273, 184, 326, 371, 179, 220, 362, 296, 158, 159, 309, 29], [350, 353, 23, 33, 284, 287, 349, 351, 126, 36, 147, 342, 360, 206, 374, 145, 345, 341, 132, 357], [33, 34, 31, 206, 37, 64, 287, 350, 36, 35, 28, 25, 351, 40, 144, 207, 353, 126, 30, 147, 360, 284, 342, 62, 78, 24, 292, 21, 175, 374, 308, 39, 379, 212, 319, 61, 23, 205, 349, 215, 27, 204, 80, 288, 325, 285, 286, 253, 22, 227, 73, 217, 216, 71, 67, 283, 324, 305, 56, 38, 123, 203, 168, 132, 145, 307, 15, 143, 70, 323, 335, 279, 352, 357, 299, 157, 341, 314, 155, 347, 345, 356, 281, 346, 43, 306, 161, 124, 386, 44, 294, 289, 113, 177, 69, 176, 5, 127, 343, 340, 369, 313, 208, 274, 165, 74, 300, 320, 330, 246, 57, 202, 153, 333, 327, 120, 304, 156, 251, 387, 293, 174, 334, 173, 96, 209, 115, 254, 383, 260, 180, 384, 331, 136, 79, 295, 146, 84, 164, 355, 311, 348, 178, 291, 392, 298, 167, 297, 339, 65, 200, 160, 296, 310, 303, 328, 290, 171, 201, 97, 395, 197, 226, 396, 135, 318, 8, 105, 93, 219, 309, 154, 117, 385, 99, 235, 20, 329, 83, 77, 158, 326, 170, 232, 58, 159, 152, 86, 87, 172], [350, 287, 33, 284, 353, 288, 342, 126, 351, 147, 206, 360, 281, 349, 144, 28, 357, 31, 352, 341, 207, 335, 331, 374, 347, 345, 286, 155, 285, 35, 356, 343, 34, 132, 325, 283, 36, 386, 379, 215, 24, 64, 346, 292, 333, 340, 330, 307, 157, 15, 348, 300, 205, 246, 145, 204, 40, 387, 37, 216, 355, 294, 27, 39, 21, 293, 212, 78, 153, 384, 324, 124, 208, 69, 120, 25, 327, 313, 334, 253, 38, 203, 30, 304, 289, 392, 160, 156, 115, 297, 62, 143, 127, 299, 146, 305, 217, 73, 175, 339, 61, 319, 23, 67, 320, 279, 123, 328, 295, 395, 383, 113, 22, 176, 306, 344, 296, 197, 291, 298, 80, 154, 167, 200, 71, 202, 201, 56, 136, 44, 159, 105, 274, 323, 70, 117, 227, 168, 290, 311, 151, 396, 393, 152, 161, 96, 308, 209, 57, 310, 164, 74, 314, 47, 81, 116, 282, 338, 5, 362, 7, 219, 8, 326, 149, 93, 388, 378, 399, 110, 79, 150, 29, 303, 174, 312, 260, 165, 332, 369, 135, 77, 52, 111, 119, 211, 177, 252, 125, 158, 84, 48, 235, 226, 385, 336, 377, 138, 381, 173], [350, 206, 353, 351, 62, 207, 349, 33, 215, 28, 35, 360, 281, 31, 374, 345, 342, 34, 64, 204, 15, 40, 379, 352, 308, 37, 24, 208, 287, 341, 347, 346, 357, 216, 203, 78, 126, 36, 71, 284, 73, 25, 212, 288, 30, 343, 205, 324, 144, 61, 305, 320, 69, 22, 356, 38, 209, 323, 327, 335, 217, 80, 147, 202, 74, 39, 292, 283, 8, 123, 253, 348, 132, 167, 79, 120, 70, 44, 21, 23, 307, 285, 286, 115, 27, 319, 175, 294, 219, 57, 300, 355, 362, 328, 218, 313, 246, 5, 176, 113, 369, 67, 235, 227, 314, 310, 211, 340, 124, 65, 299, 157, 20, 145, 386, 293, 56, 311, 201, 127]], [[75, 284, 87, 66, 65, 139, 133, 287, 67, 79, 80, 85, 84, 69, 73, 71, 74, 138, 295, 78, 125, 134, 77, 179, 119, 64, 106, 72, 144, 126, 344, 99, 86, 61, 59, 89, 135, 129, 307, 131, 169, 306, 94, 365, 304, 147, 379, 63, 244, 70, 113, 265, 347, 53, 286, 160, 140, 76, 385, 93, 155, 364, 118, 164, 132, 303, 245, 313, 319, 127, 83, 95, 98, 109, 122, 62, 123, 100, 247, 312], [66, 87, 67, 134, 133, 139, 119, 73, 74, 53, 79, 99, 65, 75, 76, 287, 72, 63, 125, 284, 131, 71, 138, 113, 106, 126, 365, 94, 86, 379, 64, 59, 114, 85, 364, 179, 51, 84, 77, 313, 307, 265, 80, 69, 359, 118, 319, 78, 264, 46, 344, 83, 306, 243, 244, 147, 303, 47, 54, 123, 279, 286, 58, 311, 111, 107, 378, 135, 245, 127, 314, 295, 45, 347, 44, 374, 93, 61, 343, 333, 43, 129, 144, 70, 304, 367, 312, 140, 239, 234, 274, 340, 167, 263, 143, 354, 238, 156, 247, 363, 293, 253, 259, 89, 320, 233, 385, 96, 104, 109, 160, 60, 278, 49, 130, 164, 258, 55, 68, 256, 103, 273, 301, 166, 132, 283, 203, 375, 62, 369, 345, 7, 50, 174, 146, 98, 157, 231, 159, 154, 169, 325, 153, 326, 205, 163, 173, 122, 248, 275, 267, 305, 327, 27, 204, 384, 331, 100, 52, 124, 315, 294, 105, 120, 14, 56, 358, 57, 317, 227, 95, 178, 396, 339, 185, 110, 255, 398, 296, 376], [66, 133, 75, 67, 134, 284, 139, 79, 87, 73, 74, 71, 65, 119, 287, 64, 131, 138, 80, 59, 72, 78, 69, 77, 125, 244, 63, 84, 53, 365, 344, 179, 113, 99, 70, 126, 114, 265, 76, 58, 295, 86, 51, 379, 384, 364, 61, 54, 247, 313, 144, 147, 264, 118, 106, 307, 333, 94, 140, 83, 85, 135, 245, 68, 304, 160, 286, 44, 243, 359, 312, 130, 46, 303, 347, 93, 306, 279, 319, 47, 154, 239, 320, 49, 7, 129, 340, 343, 45, 273, 363, 256, 293, 111, 159, 164, 274, 55, 374, 166, 378, 385, 314, 43, 127, 123, 104, 153, 204, 234, 311, 60, 107, 253, 283, 163, 169, 89, 301, 248, 167, 315, 263, 396, 317, 50, 238, 100, 184, 109, 354, 132, 174, 387, 143, 157, 96, 278, 155, 146, 124, 345, 294, 367, 173, 325, 52, 326, 156, 331, 203, 327, 233, 375, 57, 178, 103, 267, 310, 196, 259, 300, 150, 62, 296, 275, 369, 95, 14, 105, 1, 207, 255, 98, 187, 137, 288, 258, 368, 120, 373, 330, 321, 316, 216, 383, 324, 205, 358, 346, 388, 88, 4, 224, 19, 145, 309, 112, 110, 393], [66, 67, 133, 87, 126, 73, 134, 71, 139, 106, 284, 64, 69, 72, 75, 79, 74, 138, 61, 131, 80, 63, 65, 78, 287, 307, 119, 244, 86, 306, 76, 77, 312, 113, 313, 293, 344, 129, 379, 99, 84, 107, 364, 104, 144, 46, 127, 295, 53, 47, 304, 245, 365, 286, 125, 44, 7, 247, 179, 147, 279, 93, 333, 319, 359, 166, 265, 114, 303, 14, 123, 160, 94, 264, 301, 167, 347, 118, 70, 85, 68, 43, 59, 239, 311, 19, 109, 274, 135, 62, 374, 52, 164, 234, 256, 140, 132, 259, 378, 153, 255, 258, 384, 314, 340, 111, 327, 146, 367, 320, 233, 103, 248, 51, 326, 204, 122, 385, 173, 273, 130, 124, 156, 49, 89, 354, 300, 278, 238, 294], [74, 80, 66, 75, 134, 67, 284, 139, 79, 77, 71, 133, 73, 87, 65, 119, 160, 72, 179, 78, 344, 384, 379, 138, 287, 106, 244, 126, 70, 64, 131, 295, 53, 59, 264, 61, 69, 144, 364, 63, 76, 84, 365, 99, 154, 307, 313, 378, 113, 147, 239, 125, 140, 279, 319, 68, 359, 320, 54, 60, 114, 274, 7, 234, 300, 51, 312, 94, 314, 155, 374, 265, 303, 159, 327, 247, 127, 107, 306, 204, 317, 294, 304, 123, 153, 174, 129, 248, 301, 58, 340, 311, 293, 47, 275, 118, 243, 86, 278, 388, 148, 326, 57, 368, 375, 130, 156, 286, 354, 150, 46, 137, 347, 333, 288, 336, 85, 104, 88, 166, 343, 50, 43, 100, 268, 238, 385, 93, 167, 135, 256, 111, 316, 259, 83, 184, 132, 44, 387, 324, 363, 14, 173, 95, 157, 143, 398, 178, 169, 151, 283, 233, 164, 1, 369, 45, 396, 337, 62, 245, 89, 19, 367, 358, 298, 109, 124, 55, 323, 321]], [[34, 54, 103, 122, 104, 123, 93, 33, 173, 51, 124, 127, 153, 64, 24, 133, 67, 141, 193, 192, 42, 1, 81, 5, 134, 114, 194, 171, 125, 14, 37, 174, 45, 94, 58, 142, 2, 44, 172, 55, 100, 138, 15, 3, 181, 84, 60, 195, 13, 41, 61, 184, 46, 79, 38, 176, 107, 166, 47, 63, 163, 175, 43, 23, 82, 68, 154, 9, 159, 169, 131, 32, 31, 178, 180, 139, 74, 80, 129, 108, 92, 200, 11, 156, 65, 197, 49, 110, 118, 165, 111, 39, 161, 130, 121, 30, 115, 4, 186, 148, 12, 28, 188, 132, 158, 102, 137, 56, 144, 70, 199, 140, 177, 182, 101, 72, 105, 190, 57, 52, 25, 17, 91, 76, 155, 179, 10, 128, 160, 62, 89, 18, 157, 53, 191, 167, 183, 99, 21, 117, 97, 71, 77, 113, 151, 27, 120, 50, 78, 112], [54, 34, 122, 153, 103, 123, 173, 33, 5, 171, 127, 141, 133, 42, 124, 138, 51, 93, 64, 1, 81, 37, 172, 125, 67, 44, 24, 193, 46, 104, 142, 45, 134, 2, 192, 114, 175, 174, 14, 47, 9, 41, 176, 181, 60, 84, 49, 43, 180, 178, 55, 3, 61, 159, 179, 31, 186, 184, 15, 38, 58, 144, 194, 23, 121, 82, 4, 100, 63, 188, 39, 154, 13, 128, 50, 48, 163, 195, 177, 139, 131, 94, 92, 197, 77, 107, 155, 32, 143, 72, 120, 190, 118, 140, 68, 169, 52, 21, 156, 28, 70, 35, 129, 74, 71, 78, 137, 145, 56, 108, 87, 182, 151, 30, 132, 164, 102, 75, 40, 166, 106, 97, 20, 76, 17, 185, 8, 101, 62, 157, 158, 110, 183, 25, 115, 27, 161, 18, 165, 12, 146, 99, 86, 79, 88, 167, 11, 126, 136, 6, 200, 69, 111, 91, 113, 109, 53, 26, 198, 85, 148, 130, 57, 65, 199, 29, 149, 105, 189, 96], [34, 153, 54, 103, 33, 123, 122, 124, 104, 93, 64, 141, 173, 94, 127, 133, 5, 24, 51, 134, 194, 193, 1, 114, 14, 125, 42, 81, 67, 45, 142, 171, 2, 3, 37, 60, 138, 23, 174, 184, 192, 159, 58, 80, 9, 84, 195, 44, 172, 175, 47, 61, 55, 129, 31, 70, 154, 46, 144, 166, 186, 108, 100, 13, 32, 163, 4, 156, 41, 128, 63, 79, 43, 181, 118, 188, 199, 148, 107, 176, 121, 15, 99, 25, 92, 16, 180, 101, 106, 35, 147, 50, 17, 74, 111, 197, 39, 190, 169, 21, 49, 89, 191, 120, 91, 82, 139, 72, 52, 178, 30, 157, 158, 182, 77, 132, 177, 140, 56, 62, 150, 68, 102, 97, 11, 161, 165, 116, 38, 76, 36, 19, 143, 71, 155, 26, 145, 179, 183, 170, 200, 28, 137, 167, 59, 57, 198, 168, 110, 115, 96, 113, 65, 105, 185, 160, 189, 12, 131, 164, 75, 151, 69, 8, 22, 136, 149, 152, 40, 18, 27, 95, 130, 7, 112, 196, 10, 119, 48, 146, 90, 53, 88, 162, 187, 85, 66, 83, 73, 109, 126, 87, 86, 29, 98, 117, 6, 20, 135, 78], [34, 54, 122, 104, 124, 103, 153, 51, 123, 33, 93, 5, 173, 125, 193, 127, 141, 2, 64, 94, 171, 1, 24, 67, 195, 194, 42, 133, 45, 14, 134, 114, 192, 81, 100, 47, 142, 3, 46, 138, 172, 175, 156, 39, 37, 55, 30, 60, 61, 41, 166, 49, 44, 23, 84, 174, 13, 31, 79, 181, 154, 15, 159, 180, 129, 184, 163, 80, 150, 58, 72, 148, 91, 9, 200, 106, 101, 11, 144, 43, 179, 4, 35, 25, 82, 92, 32, 70, 186, 169, 176, 131, 191, 118, 128, 110, 140, 74, 116, 182, 21, 38, 188, 197, 196, 167, 178, 76, 63, 52, 96, 90, 107, 190, 149, 17, 108, 69, 71, 105, 102, 157, 68, 50, 77, 139, 155, 115, 147, 75, 111, 99, 120, 130, 12, 10, 95, 183, 132, 160, 16, 170, 185, 161, 48, 143, 62, 66, 56, 198, 29, 36, 121, 89, 189, 26, 78, 7, 151, 146, 87, 59, 164, 98, 28, 57, 158, 145, 137, 86, 168, 199, 19, 112, 177, 27, 22, 113, 165, 187, 97, 73, 53, 83, 109, 20, 119, 18, 85, 117, 65, 152, 126, 8, 40, 6, 162, 135, 136, 88], [34, 54, 123, 104, 103, 64, 133, 33, 153, 173, 122, 124, 93, 127, 51, 134, 24, 14, 114, 94, 2, 141, 174, 194, 193, 45, 3, 184, 67, 1, 42, 58, 125, 81, 46, 5, 80, 171, 100, 159, 37, 195, 172, 181, 47, 84, 41, 44, 60, 154, 142, 156, 175, 116, 192, 166, 61, 101, 79, 91, 150, 148, 186, 138, 13, 9, 35, 39, 23, 191, 55, 129, 118, 200, 49, 161, 4, 139, 180, 163, 43, 30, 63, 183, 75, 22, 76, 74, 169, 178, 15, 176, 144, 72, 10, 16, 182, 50, 140, 107, 131, 82, 99, 197, 151, 77, 11, 179, 149, 36, 31, 57, 177, 160, 126, 96, 32, 146, 71, 89, 68, 164, 190, 52, 38, 97, 66, 137, 199, 135, 132, 18, 128, 106, 147, 121, 19, 48, 110, 90, 120, 165, 152, 92, 119, 189, 28, 70, 7, 188, 167, 20, 53, 40, 157, 105, 21, 25, 155, 117, 102, 29, 62, 83, 111, 196, 185, 143, 73, 158, 88, 108, 198, 87, 78, 17, 95, 112, 65, 69]]]

# 验证 para 维度与特征组匹配
GROUP_DIMS = [168, 400, 400, 200]  # CTD, PSSM-co, DPC-PSSM, PSSM-AC
for i in range(4):
    for j in range(5):
        max_idx = max(PARA[i][j])
        assert max_idx <= GROUP_DIMS[i], \
            f'PARA[{i}][{j}] max index {max_idx} > group dim {GROUP_DIMS[i]}'
    print(f'Group {i+1} ({GROUP_NAMES[i]}): para lens={[len(PARA[i][j]) for j in range(5)]}')


def select_features_by_para(X, para_indices):
    """使用 para 硬编码索引选择特征（与原始代码一致: para.sort(); feature[idx-1]）。
    para_indices: 1-based 索引列表。
    """
    sorted_idx = sorted(para_indices)
    # 转为 0-based
    zero_based = [idx - 1 for idx in sorted_idx]
    return X[:, zero_based]


# --- Step 2: 5 次平衡采样 + 硬编码 para 特征选择 + SVM GridSearchCV ---
rng = np.random.RandomState(SEED)
pos_idx = np.where(y_train_all == 1)[0]
neg_idx = np.where(y_train_all == 0)[0]
n_minority = min(len(pos_idx), len(neg_idx))
print(f'\nPositive: {len(pos_idx)}, Negative: {len(neg_idx)}, Balanced size: {2*n_minority}')

# SVM 超参数网格（对齐原文的参数优化）
SVM_PARAM_GRID = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
}

# 存储: ensemble[group_i][sampling_j] = (para_indices, svm_model)
ensemble = [[None] * N_SAMPLINGS for _ in range(4)]
svm_best_params = {}

for j in range(N_SAMPLINGS):
    # 随机欠采样多数类
    if len(pos_idx) < len(neg_idx):
        sampled_neg = rng.choice(neg_idx, size=n_minority, replace=False)
        balanced_idx = np.concatenate([pos_idx, sampled_neg])
    else:
        sampled_pos = rng.choice(pos_idx, size=n_minority, replace=False)
        balanced_idx = np.concatenate([sampled_pos, neg_idx])
    rng.shuffle(balanced_idx)
    y_balanced = y_train_all[balanced_idx]

    for i in range(4):
        X_balanced = train_groups_scaled[i][balanced_idx]
        X_selected = select_features_by_para(X_balanced, PARA[i][j])

        # SVM GridSearchCV（第一次采样时搜索最优超参数，后续复用）
        if j == 0:
            grid = GridSearchCV(
                SVC(kernel='rbf', probability=True, random_state=SEED),
                SVM_PARAM_GRID, cv=3, scoring='roc_auc', n_jobs=-1,
            )
            grid.fit(X_selected, y_balanced)
            best_C = grid.best_params_['C']
            best_gamma = grid.best_params_['gamma']
            svm_best_params[i] = (best_C, best_gamma)
            print(f'  Group {i+1} best SVM: C={best_C}, gamma={best_gamma}, '
                  f'CV AUC={grid.best_score_:.4f}, n_feat={X_selected.shape[1]}')
            svm = grid.best_estimator_
        else:
            best_C, best_gamma = svm_best_params[i]
            svm = SVC(kernel='rbf', C=best_C, gamma=best_gamma,
                      probability=True, random_state=SEED)
            svm.fit(X_selected, y_balanced)

        ensemble[i][j] = (PARA[i][j], svm)

    print(f'Sampling {j+1}/{N_SAMPLINGS}: trained 4 SVMs (balanced n={len(balanced_idx)})')

print(f'\nTotal ensemble: {4 * N_SAMPLINGS} SVMs')

Group 1 (CTD): scaled, dim=168
Group 2 (PSSM-co): scaled, dim=400
Group 3 (DPC-PSSM): scaled, dim=400
Group 4 (PSSM-AC): scaled, dim=200
Group 1 (CTD): para lens=[90, 120, 110, 160, 30]
Group 2 (PSSM-co): para lens=[190, 20, 190, 200, 120]
Group 3 (DPC-PSSM): para lens=[80, 180, 200, 130, 170]
Group 4 (PSSM-AC): para lens=[150, 170, 200, 200, 180]

Positive: 205, Negative: 902, Balanced size: 410
  Group 1 best SVM: C=1.0, gamma=scale, CV AUC=0.7815, n_feat=90
  Group 2 best SVM: C=1.0, gamma=scale, CV AUC=0.8997, n_feat=190
  Group 3 best SVM: C=100.0, gamma=0.1, CV AUC=0.9221, n_feat=80
  Group 4 best SVM: C=1.0, gamma=auto, CV AUC=0.8948, n_feat=150
Sampling 1/5: trained 4 SVMs (balanced n=410)
Sampling 2/5: trained 4 SVMs (balanced n=410)
Sampling 3/5: trained 4 SVMs (balanced n=410)
Sampling 4/5: trained 4 SVMs (balanced n=410)
Sampling 5/5: trained 4 SVMs (balanced n=410)

Total ensemble: 20 SVMs


## 5. 集成预测与评估

对齐 `predict_main.py` 的 `perform_prediction`：20 个模型概率取平均。

In [10]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        m = ids == b
        if np.any(m):
            ece += (np.sum(m) / n) * abs(float(np.mean(y_true[m])) - float(np.mean(y_prob[m])))
    return float(ece)


def evaluate_binary_full(y_true, y_prob, threshold=0.5):
    y_cls = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_cls).ravel()
    return {
        'AUC':  float(roc_auc_score(y_true, y_prob)),
        'AUPRC': float(average_precision_score(y_true, y_prob)),
        'F1':   float(f1_score(y_true, y_cls)),
        'MCC':  float(matthews_corrcoef(y_true, y_cls)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'ECE':  expected_calibration_error(y_true, y_prob),
        'ACC':  float(accuracy_score(y_true, y_cls)),
        'SN':   float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
        'SP':   float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        'Threshold': float(threshold),
    }


# --- 集成预测（对齐 predict_main.py 的 perform_prediction） ---
def ensemble_predict(groups_scaled, ensemble_models):
    """20 个模型概率取平均。使用 para 硬编码索引进行特征选择。"""
    n_samples = groups_scaled[0].shape[0]
    pred_sum = np.zeros(n_samples)
    count = 0
    for i in range(4):
        for j in range(N_SAMPLINGS):
            para_indices, svm = ensemble_models[i][j]
            X_sel = select_features_by_para(groups_scaled[i], para_indices)
            pred_sum += svm.predict_proba(X_sel)[:, 1]
            count += 1
    return pred_sum / count


y_prob = ensemble_predict(test_groups_scaled, ensemble)

# 用训练集的一个 hold-out 来找阈值
_, hold_idx = train_test_split(
    np.arange(len(y_train_all)), test_size=0.1, stratify=y_train_all, random_state=SEED
)
val_prob = ensemble_predict([g[hold_idx] for g in train_groups_scaled], ensemble)
y_val = y_train_all[hold_idx]

best_thr, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.9, 0.01):
    f = f1_score(y_val, (val_prob >= t).astype(int))
    if f > best_f1:
        best_f1, best_thr = f, t
print(f'Best threshold (val F1={best_f1:.4f}): {best_thr:.2f}')

metrics = evaluate_binary_full(y_test, y_prob, threshold=best_thr)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')

Best threshold (val F1=0.9756): 0.74
AUC: 0.9430
AUPRC: 0.6862
F1: 0.5333
MCC: 0.5017
Brier: 0.1164
ECE: 0.2398
ACC: 0.9266
SN: 0.4615
SP: 0.9731
Threshold: 0.7400


In [11]:
result = {'method': 'AcrPred (SVM)', 'metrics': metrics}
with open(f'{RESULTS_DIR}/acrpred_metrics.json', 'w') as f:
    json.dump(result, f, indent=2)
np.savez(f'{RESULTS_DIR}/acrpred_predictions.npz', y_true=y_test, y_prob=y_prob)
print('Results saved.')

Results saved.
